# 02 · Procesamiento: limpieza, países de destino e indicadores

Este notebook toma los extractos del notebook 01 y prepara:

1. **Producto y tipo de producto** de cada subpartida (por ejemplo, café verde o tostado, cacao en grano o manteca, madera aserrada).
2. **País de destino normalizado**: código ISO-3 para el mapa, nombre en español y pertenencia a la **UE-27**.
3. **Indicadores anuales**: FOB, volumen, participación de la UE, valor unitario (US$/kg) y variación interanual.
4. Los archivos del dashboard (`web/data/datos.json`).

**Periodo comparable.** El último año (2026) solo tiene los meses publicados. Para compararlo con años anteriores se calcula, en todos los
años, el acumulado de enero hasta ese mismo mes (columnas `*_p`).

In [1]:
import json
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ORIGINALES = RAIZ / "datos" / "originales"
PROCESADOS = RAIZ / "datos" / "procesados"
WEB_DATA = RAIZ / "web" / "data"
PROCESADOS.mkdir(parents=True, exist_ok=True)
WEB_DATA.mkdir(parents=True, exist_ok=True)

eudr = pd.read_csv(ORIGINALES / "extracto_g8_eudr_mensual.csv", dtype={"subpartida": str})
total_pais = pd.read_csv(ORIGINALES / "extracto_g8_total_pais_mensual.csv")
metadatos = json.loads((ORIGINALES / "metadatos_descarga.json").read_text(encoding="utf-8"))
print(eudr.shape, total_pais.shape)
eudr.head()

(62535, 8) (33377, 6)


,anio,mes,pais_sunat,subpartida,descripcion,fob_usd,peso_neto_kg,peso_bruto_kg
0,2005,1,ESTADOS UNIDOS,0901110000,"CAFE SIN DESCAFEINAR, SIN TOSTAR",8695463.95,4576689.0,4623134.30
1,2005,2,ESTADOS UNIDOS,0901110000,"CAFE SIN DESCAFEINAR, SIN TOSTAR",4485982.33,2391402.0,2413291.75
2,2005,3,ESTADOS UNIDOS,0901110000,"CAFE SIN DESCAFEINAR, SIN TOSTAR",1591614.01,904797.0,913740.00
3,2005,4,ESTADOS UNIDOS,0901110000,"CAFE SIN DESCAFEINAR, SIN TOSTAR",701375.14,320850.0,323372.00
4,2005,5,ESTADOS UNIDOS,0901110000,"CAFE SIN DESCAFEINAR, SIN TOSTAR",4416867.84,1851140.0,1869092.00


## 1. Control de calidad

In [2]:
print("Duplicados exactos:", eudr.duplicated().sum())
print("Duplicados por clave (año, mes, país, subpartida):", eudr.duplicated(["anio", "mes", "pais_sunat", "subpartida"]).sum())
print("Valores negativos:", (eudr[["fob_usd", "peso_neto_kg"]] < 0).sum().to_dict())
print("Filas con FOB > 0 y peso neto = 0:", ((eudr.fob_usd > 0) & (eudr.peso_neto_kg == 0)).sum())
ANIO_PARCIAL = int(eudr.anio.max())
ULTIMO_MES = int(eudr.loc[eudr.anio == ANIO_PARCIAL, "mes"].max())
print(f"Último dato: {ANIO_PARCIAL}, mes {ULTIMO_MES}")

Duplicados exactos: 0
Duplicados por clave (año, mes, país, subpartida): 0
Valores negativos: {'fob_usd': 0, 'peso_neto_kg': 0}
Filas con FOB > 0 y peso neto = 0: 0
Último dato: 2026, mes 8


Una misma subpartida puede aparecer dos veces para un país y mes cuando SUNAT lista el país con dos nombres. Por ejemplo, "FRANCIA"
y "FRANCE, METROPOLITAN", o "COSTA DE MARFIL" y "COTE D'IVOIRE". Esos casos se unifican al asignar el código ISO-3 y luego se suman.

## 2. Países: código ISO-3, nombre y bloque UE-27

SUNAT usa nombres propios. Se arma una tabla de equivalencias a mano.
- Los departamentos franceses de ultramar (Guadalupe, Martinica, Guayana Francesa, Reunión, Mayotte) se suman a **Francia**: son territorio
  de la UE.
- "Escocia" se suma al **Reino Unido**, y "Serbia y Montenegro" y "Yugoslavia" a **Serbia**.
- Los destinos sin país (aguas internacionales, no declarados, zonas francas del Perú) llevan un código que empieza con `X` y no van al
  mapa.
- La UE se define con sus **27 miembros actuales** en todos los años, para que la serie sea comparable. El Reino Unido queda fuera también
  antes del Brexit.

In [3]:
PAISES = {
    "AFGANISTAN": ("AFG", "Afganistán"), "AGUAS INTERNACIONALES": ("XAI", "Aguas internacionales"), "ALBANIA": ("ALB", "Albania"),
    "ALEMANIA": ("DEU", "Alemania"), "ANDORRA": ("AND", "Andorra"), "ANGOLA": ("AGO", "Angola"), "ANGUILLA": ("AIA", "Anguila"),
    "ANTIGUA Y BARBUDA": ("ATG", "Antigua y Barbuda"), "ANTILLAS HOLANDESAS": ("XAN", "Antillas Neerlandesas (extinta)"),
    "ARABIA SAUDITA": ("SAU", "Arabia Saudita"), "ARGELIA": ("DZA", "Argelia"), "ARGENTINA": ("ARG", "Argentina"),
    "ARMENIA": ("ARM", "Armenia"), "ARUBA": ("ABW", "Aruba"), "AUSTRALIA": ("AUS", "Australia"), "AUSTRIA": ("AUT", "Austria"),
    "AZERBAIJAN": ("AZE", "Azerbaiyán"), "BAHAMAS": ("BHS", "Bahamas"), "BAHREIN": ("BHR", "Baréin"), "BANGLADESH": ("BGD", "Bangladés"),
    "BARBADOS": ("BRB", "Barbados"), "BELARUS": ("BLR", "Bielorrusia"), "BELGICA": ("BEL", "Bélgica"), "BELICE": ("BLZ", "Belice"),
    "BENIN": ("BEN", "Benín"), "BERMUDAS": ("BMU", "Bermudas"), "BOLIVIA": ("BOL", "Bolivia"),
    "BONAIRE,SAN EUSTAQUIO Y SABA": ("BES", "Bonaire, San Eustaquio y Saba"), "BOSNIA-HERZEGOVINA": ("BIH", "Bosnia y Herzegovina"),
    "BOTSWANA": ("BWA", "Botsuana"), "BRASIL": ("BRA", "Brasil"), "BRUNEI DARUSSALAM": ("BRN", "Brunéi"), "BULGARIA": ("BGR", "Bulgaria"),
    "BURKINA FASO": ("BFA", "Burkina Faso"), "BURUNDI": ("BDI", "Burundi"), "BUTAN": ("BTN", "Bután"), "CABO VERDE": ("CPV", "Cabo Verde"),
    "CAIMAN,ISLAS": ("CYM", "Islas Caimán"), "CAMBOYA": ("KHM", "Camboya"), "CAMERUN,REPUBLICA UNIDA DEL": ("CMR", "Camerún"),
    "CANADA": ("CAN", "Canadá"), "CHAD": ("TCD", "Chad"), "CHILE": ("CHL", "Chile"), "CHINA": ("CHN", "China"), "CHIPRE": ("CYP", "Chipre"),
    "COCOS (KEELING),ISLAS": ("CCK", "Islas Cocos"), "COLOMBIA": ("COL", "Colombia"), "COMORAS": ("COM", "Comoras"),
    "COOK, ISLAS": ("COK", "Islas Cook"), "COREA (NORTE), REPUBLICA POPULAR DEMOCRATICA DE": ("PRK", "Corea del Norte"),
    "COREA (SUR), REPUBLICA DE": ("KOR", "Corea del Sur"), "COSTA DE MARFIL": ("CIV", "Costa de Marfil"), "COSTA RICA": ("CRI", "Costa Rica"),
    "COTE D'IVOIRE": ("CIV", "Costa de Marfil"), "CROACIA": ("HRV", "Croacia"), "CUBA": ("CUB", "Cuba"), "CURACAO": ("CUW", "Curazao"),
    "DINAMARCA": ("DNK", "Dinamarca"), "DJIBOUTI": ("DJI", "Yibuti"), "DOMINICA": ("DMA", "Dominica"), "ECUADOR": ("ECU", "Ecuador"),
    "EGIPTO": ("EGY", "Egipto"), "EL SALVADOR": ("SLV", "El Salvador"), "EMIRATOS ARABES UNIDOS": ("ARE", "Emiratos Árabes Unidos"),
    "ERITREA": ("ERI", "Eritrea"), "ESCOCIA": ("GBR", "Reino Unido"), "ESLOVAQUIA": ("SVK", "Eslovaquia"), "ESLOVENIA": ("SVN", "Eslovenia"),
    "ESPAÑA": ("ESP", "España"), "ESTADOS UNIDOS": ("USA", "Estados Unidos"), "ESTONIA": ("EST", "Estonia"), "ETIOPIA": ("ETH", "Etiopía"),
    "FEROE, ISLAS": ("FRO", "Islas Feroe"), "FIJI": ("FJI", "Fiyi"), "FILIPINAS": ("PHL", "Filipinas"), "FINLANDIA": ("FIN", "Finlandia"),
    "FRANCE, METROPOLITAN": ("FRA", "Francia"), "FRANCIA": ("FRA", "Francia"), "GABON": ("GAB", "Gabón"), "GAMBIA": ("GMB", "Gambia"),
    "GEORGIA": ("GEO", "Georgia"), "GHANA": ("GHA", "Ghana"), "GIBRALTAR": ("GIB", "Gibraltar"), "GRANADA": ("GRD", "Granada"),
    "GRECIA": ("GRC", "Grecia"), "GROENLANDIA": ("GRL", "Groenlandia"), "GUADALUPE": ("FRA", "Francia"), "GUAM": ("GUM", "Guam"),
    "GUATEMALA": ("GTM", "Guatemala"), "GUAYANA FRANCESA": ("FRA", "Francia"), "GUINEA": ("GIN", "Guinea"),
    "GUINEA ECUATORIAL": ("GNQ", "Guinea Ecuatorial"), "GUINEA-BISSAU": ("GNB", "Guinea-Bisáu"), "GUYANA": ("GUY", "Guyana"),
    "HAITI": ("HTI", "Haití"), "HONDURAS": ("HND", "Honduras"), "HONG KONG": ("HKG", "Hong Kong"), "HUNGRIA": ("HUN", "Hungría"),
    "INDIA": ("IND", "India"), "INDONESIA": ("IDN", "Indonesia"), "IRAK": ("IRQ", "Irak"), "IRAN, REPUBLICA ISLAMICA DEL": ("IRN", "Irán"),
    "IRLANDA (EIRE)": ("IRL", "Irlanda"), "ISLANDIA": ("ISL", "Islandia"), "ISRAEL": ("ISR", "Israel"), "ITALIA": ("ITA", "Italia"),
    "JAMAICA": ("JAM", "Jamaica"), "JAPON": ("JPN", "Japón"), "JORDANIA": ("JOR", "Jordania"), "KAZAJSTAN": ("KAZ", "Kazajistán"),
    "KENIA": ("KEN", "Kenia"), "KIRGUIZISTAN": ("KGZ", "Kirguistán"), "KIRIBATI": ("KIR", "Kiribati"), "KUWAIT": ("KWT", "Kuwait"),
    "LAOS, REPUBLICA POPULAR DEMOCRATICA DE": ("LAO", "Laos"), "LETONIA": ("LVA", "Letonia"), "LIBANO": ("LBN", "Líbano"),
    "LIBERIA": ("LBR", "Liberia"), "LIBIA": ("LBY", "Libia"), "LIECHTENSTEIN": ("LIE", "Liechtenstein"), "LITUANIA": ("LTU", "Lituania"),
    "LUXEMBURGO": ("LUX", "Luxemburgo"), "MACAO": ("MAC", "Macao"), "MACEDONIA F.Y.R.": ("MKD", "Macedonia del Norte"),
    "MADAGASCAR": ("MDG", "Madagascar"), "MALAWI": ("MWI", "Malaui"), "MALAYSIA": ("MYS", "Malasia"), "MALDIVAS": ("MDV", "Maldivas"),
    "MALI": ("MLI", "Malí"), "MALTA": ("MLT", "Malta"), "MARIANAS DEL NORTE, ISLAS": ("MNP", "Islas Marianas del Norte"),
    "MARRUECOS": ("MAR", "Marruecos"), "MARSHALL, ISLAS": ("MHL", "Islas Marshall"), "MARTINICA": ("FRA", "Francia"),
    "MAURICIO": ("MUS", "Mauricio"), "MAURITANIA": ("MRT", "Mauritania"), "MAYOTTE": ("FRA", "Francia"), "MEXICO": ("MEX", "México"),
    "MICRONESIA, ESTADOS FEDERADOS DE": ("FSM", "Micronesia"), "MOLDAVIA": ("MDA", "Moldavia"), "MONACO": ("MCO", "Mónaco"),
    "MONGOLIA": ("MNG", "Mongolia"), "MONTENEGRO": ("MNE", "Montenegro"), "MONTSERRAT, ISLA": ("MSR", "Montserrat"),
    "MOZAMBIQUE": ("MOZ", "Mozambique"), "MYANMAR": ("MMR", "Myanmar"), "NAMIBIA": ("NAM", "Namibia"), "NAURU": ("NRU", "Nauru"),
    "NAVIDAD (CHRISTMAS), ISLA": ("CXR", "Isla de Navidad"), "NEPAL": ("NPL", "Nepal"), "NICARAGUA": ("NIC", "Nicaragua"),
    "NIGER": ("NER", "Níger"), "NIGERIA": ("NGA", "Nigeria"), "NO DECLARADOS": ("XND", "No declarado"), "NORFOLK, ISLA": ("NFK", "Isla Norfolk"),
    "NORUEGA": ("NOR", "Noruega"), "NUEVA CALEDONIA": ("NCL", "Nueva Caledonia"), "NUEVA ZELANDA": ("NZL", "Nueva Zelanda"),
    "OMAN": ("OMN", "Omán"), "PAISES BAJOS": ("NLD", "Países Bajos"),
    "PAISES NO PRECISADOS EN OTRA PARTE Y DESCONOCIDOS": ("XNP", "País no precisado"), "PAKISTAN": ("PAK", "Pakistán"),
    "PALAU, ISLAS": ("PLW", "Palaos"), "PANAMA": ("PAN", "Panamá"), "PAPUASIA NUEVA GUINEA": ("PNG", "Papúa Nueva Guinea"),
    "PARAGUAY": ("PRY", "Paraguay"), "PERÚ": ("XPE", "Perú (destino nacional)"), "POLINESIA FRANCESA": ("PYF", "Polinesia Francesa"),
    "POLONIA": ("POL", "Polonia"), "PORTUGAL": ("PRT", "Portugal"), "PUERTO RICO": ("PRI", "Puerto Rico"), "QATAR": ("QAT", "Catar"),
    "REINO UNIDO": ("GBR", "Reino Unido"), "REPUBLICA CENTROAFRICANA": ("CAF", "República Centroafricana"),
    "REPUBLICA CHECA": ("CZE", "Chequia"), "REPUBLICA DOMINICANA": ("DOM", "República Dominicana"),
    "REPÚBLICA DEL CONGO": ("COG", "República del Congo"), "REPÚBLICA DEMOCRÁTICA DEL CONGO": ("COD", "R. D. del Congo"),
    "REUNION": ("FRA", "Francia"), "RUANDA": ("RWA", "Ruanda"), "RUMANIA": ("ROU", "Rumania"), "RUSIA": ("RUS", "Rusia"),
    "SALOMON, ISLAS": ("SLB", "Islas Salomón"), "SAMOA": ("WSM", "Samoa"), "SAMOA NORTEAMERICANA": ("ASM", "Samoa Americana"),
    "SAN BARTOLOME": ("BLM", "San Bartolomé"), "SAN CRISTOBAL Y NIEVES": ("KNA", "San Cristóbal y Nieves"),
    "SAN MARINO": ("SMR", "San Marino"), "SAN MARTIN": ("MAF", "San Martín"),
    "SAN VICENTE Y LAS GRANADINAS": ("VCT", "San Vicente y las Granadinas"), "SANTA ELENA": ("SHN", "Santa Elena"),
    "SANTA LUCIA": ("LCA", "Santa Lucía"), "SANTA SEDE": ("VAT", "Santa Sede"), "SANTO TOME Y PRINCIPE": ("STP", "Santo Tomé y Príncipe"),
    "SAWSILANDIA": ("SWZ", "Esuatini"), "SENEGAL": ("SEN", "Senegal"), "SERBIA": ("SRB", "Serbia"), "SERBIA Y MONTENEGRO": ("SRB", "Serbia"),
    "SEYCHELLES": ("SYC", "Seychelles"), "SIERRA LEONA": ("SLE", "Sierra Leona"), "SINGAPUR": ("SGP", "Singapur"),
    "SIRIA, REPUBLICA ARABE DE": ("SYR", "Siria"), "SOMALIA": ("SOM", "Somalia"), "SRI LANKA": ("LKA", "Sri Lanka"),
    "SUDAFRICA, REPUBLICA DE": ("ZAF", "Sudáfrica"), "SUDAN": ("SDN", "Sudán"), "SUECIA": ("SWE", "Suecia"), "SUIZA": ("CHE", "Suiza"),
    "SURINAM": ("SUR", "Surinam"), "SVALBARD AND JAN MAYEN ISLANDS": ("SJM", "Svalbard y Jan Mayen"), "TAILANDIA": ("THA", "Tailandia"),
    "TAIWAN (FORMOSA)": ("TWN", "Taiwán"), "TANZANIA, REPUBLICA UNIDA DE": ("TZA", "Tanzania"),
    "TERRITORIO ANTARTICO BRITANICO": ("XAT", "Territorio Antártico Británico"),
    "TERRITORIO BRITANICO DEL OCEANO INDICO": ("IOT", "Territorio Británico del Océano Índico"), "TIMOR DEL ESTE": ("TLS", "Timor Oriental"),
    "TOGO": ("TGO", "Togo"), "TOKELAU": ("TKL", "Tokelau"), "TONGA": ("TON", "Tonga"), "TRINIDAD Y TOBAGO": ("TTO", "Trinidad y Tobago"),
    "TUNICIA": ("TUN", "Túnez"), "TURCAS Y CAICOS, ISLAS": ("TCA", "Islas Turcas y Caicos"), "TURKMENISTAN": ("TKM", "Turkmenistán"),
    "TURQUIA": ("TUR", "Turquía"), "UCRANIA": ("UKR", "Ucrania"), "UGANDA": ("UGA", "Uganda"),
    "UNITED STATES MINOR OUTLYING ISLANDS": ("UMI", "Islas menores de EE. UU."), "URUGUAY": ("URY", "Uruguay"),
    "UZBEKISTAN": ("UZB", "Uzbekistán"), "VANUATU": ("VUT", "Vanuatu"), "VENEZUELA": ("VEN", "Venezuela"), "VIET NAM": ("VNM", "Vietnam"),
    "VIRGENES, ISLAS (BRITANICAS)": ("VGB", "Islas Vírgenes Británicas"), "VIRGENES, ISLAS (NORTEAMERICANAS)": ("VIR", "Islas Vírgenes de EE. UU."),
    "YEMEN": ("YEM", "Yemen"), "YUGOSLAVIA": ("SRB", "Serbia"), "ZAMBIA": ("ZMB", "Zambia"), "ZIMBABWE": ("ZWE", "Zimbabue"),
    "ZONA NEUTRAL (PALESTINA)": ("PSE", "Palestina"), "ZONAS FRANCAS DEL PERU": ("XZF", "Zonas francas del Perú"),
}
UE27 = {
    "AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK", "EST", "FIN", "FRA", "DEU", "GRC", "HUN", "IRL",
    "ITA", "LVA", "LTU", "LUX", "MLT", "NLD", "POL", "PRT", "ROU", "SVK", "SVN", "ESP", "SWE",
}

sin_mapear = sorted((set(eudr.pais_sunat) | set(total_pais.pais_sunat)) - set(PAISES))
assert not sin_mapear, f"Países sin equivalencia: {sin_mapear}"
assert len(UE27) == 27

tabla_paises = (
    pd.DataFrame([(k, *v) for k, v in PAISES.items()], columns=["pais_sunat", "iso3", "pais"])
    .assign(ue27=lambda d: d.iso3.isin(UE27))
)
tabla_paises.to_csv(PROCESADOS / "tabla_paises.csv", index=False)
print("Países de la UE presentes en los datos:", sorted(set(tabla_paises.query("ue27").pais)))

Países de la UE presentes en los datos: ['Alemania', 'Austria', 'Bulgaria', 'Bélgica', 'Chequia', 'Chipre', 'Croacia', 'Dinamarca', 'Eslovaquia', 'Eslovenia', 'España', 'Estonia', 'Finlandia', 'Francia', 'Grecia', 'Hungría', 'Irlanda', 'Italia', 'Letonia', 'Lituania', 'Luxemburgo', 'Malta', 'Países Bajos', 'Polonia', 'Portugal', 'Rumania', 'Suecia']


## 3. Producto y tipo de producto (subgrupo)

Se agrupan las subpartidas en pocas categorías legibles. Para madera se juntan las partidas pequeñas del capítulo 44, así cada producto tiene
como máximo 7 tipos.

In [4]:
SUB_CACAO = {
    "1801": "Cacao en grano", "1802": "Cáscara y residuos de cacao", "1803": "Pasta de cacao",
    "1804": "Manteca de cacao", "1805": "Cacao en polvo", "1806": "Chocolate y preparaciones",
}
SUB_MADERA = {
    "4407": "Madera aserrada", "4409": "Madera perfilada (pisos, decking, molduras)", "4403": "Madera en bruto (rolliza)",
    "4402": "Carbón vegetal", "4408": "Chapas y láminas", "4410": "Tableros", "4411": "Tableros", "4412": "Tableros",
}


def clasificar(codigo: str) -> tuple[str, str]:
    if codigo.startswith("0901"):
        if codigo.startswith("090111"):
            return "Café", "Café verde"
        if codigo.startswith("090112"):
            return "Café", "Café verde descafeinado"
        if codigo.startswith("09012"):
            return "Café", "Café tostado"
        return "Café", "Cáscara, cascarilla y sucedáneos"
    if codigo.startswith("18"):
        return "Cacao", SUB_CACAO[codigo[:4]]
    partida = codigo[:4]
    if partida in SUB_MADERA:
        return "Madera", SUB_MADERA[partida]
    if partida >= "4414":
        return "Madera", "Carpintería y manufacturas"
    return "Madera", "Otros (leña, astillas, flejes, durmientes)"


clasif = pd.DataFrame([(c, *clasificar(c)) for c in eudr.subpartida.unique()], columns=["subpartida", "producto", "subgrupo"])
datos = (
    eudr.merge(clasif, on="subpartida")
    .merge(tabla_paises[["pais_sunat", "iso3", "pais", "ue27"]], on="pais_sunat", how="left")
)
assert datos.iso3.notna().all()
datos["periodo_comparable"] = datos.mes <= ULTIMO_MES  # enero..último mes publicado del año parcial
clasif.groupby(["producto", "subgrupo"]).size().rename("n_subpartidas").to_frame()

n_subpartidas
producto subgrupo                                                  
Cacao    Cacao en grano                                           4
         Cacao en polvo                                           1
         Chocolate y preparaciones                                9
         Cáscara y residuos de cacao                              1
         Manteca de cacao                                         7
         Pasta de cacao                                           2
Café     Café tostado                                             3
         Café verde                                               3
         Café verde descafeinado                                  1
         Cáscara, cascarilla y sucedáneos                         1
Madera   Carbón vegetal                                           4
         Carpintería y manufacturas                              56
         Chapas y láminas                                         5
         Madera aserrada                                         18
         Madera en bruto (rolliza)                                7
         Madera perfilada (pisos, decking, molduras)             16
         Otros (leña, astillas, flejes, durmientes)              18
         Tableros                                                35

## 4. Tabla mensual limpia

Se suman los países que SUNAT lista con dos nombres (ahora tienen el mismo ISO-3). Esta tabla es la base procesada del proyecto.

In [5]:
claves = ["anio", "mes", "producto", "subgrupo", "subpartida", "descripcion", "iso3", "pais", "ue27"]
mensual = (
    datos.groupby(claves, as_index=False)[["fob_usd", "peso_neto_kg", "peso_bruto_kg"]].sum()
    .sort_values(["anio", "mes", "producto", "fob_usd"], ascending=[True, True, True, False])
)
mensual.to_csv(PROCESADOS / "exportaciones_eudr_mensual.csv", index=False)
print(mensual.shape)
mensual.head()

(62437, 12)


,anio,mes,producto,subgrupo,subpartida,descripcion,iso3,pais,ue27,fob_usd,peso_neto_kg,peso_bruto_kg
13,2005,1,Cacao,Chocolate y preparaciones,1806320000,CHOCOLATES Y SUS PREPARAC. EN BLOQUES. TABLETA...,VEN,Venezuela,False,304728.0,131047.2,157825.4
21,2005,1,Cacao,Manteca de cacao,1804000011,MANTECA DE CACAO C/INDICE DE ACIDEZ EXPRESADO ...,FRA,Francia,True,180000.0,40000.0,40910.0
22,2005,1,Cacao,Manteca de cacao,1804000011,MANTECA DE CACAO C/INDICE DE ACIDEZ EXPRESADO ...,USA,Estados Unidos,False,149765.0,38200.0,39880.0
25,2005,1,Cacao,Manteca de cacao,1804000012,MANTECA DE CACAO C/INDICE DE ACIDEZ EXPRESADO ...,USA,Estados Unidos,False,98450.0,20000.0,20470.0
24,2005,1,Cacao,Manteca de cacao,1804000012,MANTECA DE CACAO C/INDICE DE ACIDEZ EXPRESADO ...,NLD,Países Bajos,True,78619.2,20000.0,20420.0


## 5. Indicadores anuales por producto

Para cada año y producto:
- `fob_ue`, `fob_total`: valor FOB exportado a la UE-27 y al mundo.
- `part_ue_fob`, `part_ue_kg`: participación de la UE (%) en valor y volumen.
- `vu_ue`, `vu_resto`: valor unitario (US$/kg) hacia la UE y hacia el resto del mundo.
- `var_fob_ue`: variación porcentual del FOB a la UE respecto al año anterior.

La tabla se calcula dos veces: con años completos (`periodo = "anual"`) y con enero–último mes (`periodo = "comparable"`).

In [6]:
def indicadores(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["anio", "producto", "ue27"])[["fob_usd", "peso_neto_kg"]].sum().unstack("ue27", fill_value=0)
    out = pd.DataFrame({
        "fob_ue": g[("fob_usd", True)],
        "fob_resto": g[("fob_usd", False)],
        "kg_ue": g[("peso_neto_kg", True)],
        "kg_resto": g[("peso_neto_kg", False)],
    })
    out["fob_total"] = out.fob_ue + out.fob_resto
    out["kg_total"] = out.kg_ue + out.kg_resto
    out["part_ue_fob"] = out.fob_ue / out.fob_total * 100
    out["part_ue_kg"] = out.kg_ue / out.kg_total * 100
    out["vu_ue"] = out.fob_ue / out.kg_ue
    out["vu_resto"] = out.fob_resto / out.kg_resto
    out = out.reset_index().sort_values(["producto", "anio"])
    out["var_fob_ue"] = out.groupby("producto").fob_ue.pct_change() * 100
    out["var_kg_ue"] = out.groupby("producto").kg_ue.pct_change() * 100
    return out


ind_anual = indicadores(datos[datos.anio < ANIO_PARCIAL]).assign(periodo="anual")
ind_comparable = indicadores(datos[datos.periodo_comparable]).assign(periodo="comparable")
ind = pd.concat([ind_anual, ind_comparable], ignore_index=True)
ind.to_csv(PROCESADOS / "indicadores_anuales_producto.csv", index=False)

cols = ["anio", "producto", "fob_ue", "fob_total", "part_ue_fob", "kg_ue", "vu_ue", "vu_resto", "var_fob_ue"]
ind_anual.query("anio >= 2015")[cols].round(1).pivot(index="anio", columns="producto", values=["fob_ue", "part_ue_fob"])

fob_ue                          part_ue_fob             
producto        Cacao         Café      Madera       Cacao  Café Madera
anio                                                                   
2015      174948017.9  334084174.4  14125534.0        65.6  54.4    9.7
2016      174954017.9  395421763.9  12556707.7        59.5  51.9   10.0
2017      120527914.5  365169509.7  13514383.5        50.9  51.2   11.5
2018      129407067.1  342931387.6  20309745.7        48.4  50.2   17.0
2019      126125210.3  324927929.1  23541645.1        42.7  51.0   19.5
2020      116374805.1  318832452.2  20060102.6        41.6  49.6   22.3
2021      114719211.0  395810927.3  37472882.5        37.7  51.4   31.5
2022       96714079.9  634897753.9  36480721.1        29.0  51.4   30.0
2023      128881418.3  395014195.6  24275039.5        30.1  48.2   25.1
2024      445432954.5  497096682.4  22816815.8        34.8  45.1   28.4
2025      651557585.1  727523122.8  21127615.2        40.3  40.4   32.4

## 6. Total exportado por el Perú a la UE-27 (todas las mercancías)

Sirve para medir cuánto pesan los productos EUDR en el comercio total del Perú con la UE.

In [7]:
tp = total_pais.merge(tabla_paises[["pais_sunat", "iso3", "ue27"]], on="pais_sunat")
total_ue = (
    tp[tp.ue27].assign(fob_p=lambda d: d.fob_usd.where(d.mes <= ULTIMO_MES, 0))
    .groupby("anio").agg(fob_total_ue=("fob_usd", "sum"), fob_total_ue_p=("fob_p", "sum"))
)
total_ue.to_csv(PROCESADOS / "total_exportado_ue27_anual.csv")
total_ue.tail()

,fob_total_ue,fob_total_ue_p
anio,,
2022,6.745250e+09,4.371827e+09
2023,6.924031e+09,4.348282e+09
2024,8.236559e+09,5.061648e+09
2025,1.025403e+10,6.189088e+09
2026,6.334095e+09,6.334095e+09


## 7. Datos para el dashboard (`web/data/datos.json`)

Se exportan tablas compactas (listas en vez de objetos) para que la página cargue rápido:
- `anual`: `[año, producto, iso3, fob, kg, fob_p, kg_p]`, donde `_p` es el acumulado enero–último mes.
- `sub`: `[año, subgrupo, es_UE, fob, kg, fob_p, kg_p]`.
- `total_ue`: `[año, fob total a la UE, fob total a la UE enero–último mes]`.

In [8]:
PRODUCTOS = ["Café", "Cacao", "Madera"]
datos["fob_p"] = datos.fob_usd.where(datos.periodo_comparable, 0)
datos["kg_p"] = datos.peso_neto_kg.where(datos.periodo_comparable, 0)
medidas = {"fob_usd": "sum", "peso_neto_kg": "sum", "fob_p": "sum", "kg_p": "sum"}

anual_pais = datos.groupby(["anio", "producto", "iso3"], as_index=False).agg(medidas)
anual_pais["p"] = anual_pais.producto.map(PRODUCTOS.index)

subgrupos = clasif[["producto", "subgrupo"]].drop_duplicates().sort_values(["producto", "subgrupo"])
subgrupos["p"] = subgrupos.producto.map(PRODUCTOS.index)
subgrupos = subgrupos.sort_values(["p", "subgrupo"]).reset_index(drop=True)
indice_sub = {(r.producto, r.subgrupo): i for i, r in subgrupos.iterrows()}
anual_sub = datos.groupby(["anio", "producto", "subgrupo", "ue27"], as_index=False).agg(medidas)
anual_sub["s"] = [indice_sub[(p, s)] for p, s in zip(anual_sub.producto, anual_sub.subgrupo)]


def redondear(x: float) -> float:
    return round(float(x), 1)


web = {
    "meta": {
        "actualizado": metadatos["G8"]["fecha_descarga"],
        "anio_min": int(datos.anio.min()),
        "anio_max": ANIO_PARCIAL,
        "anio_parcial": ANIO_PARCIAL,
        "ultimo_mes": ULTIMO_MES,
        "fuente": "SUNAT, Nota Tributaria, Cuadro G8",
    },
    "productos": PRODUCTOS,
    "paises": {r.iso3: [r.pais, int(r.ue27)] for r in tabla_paises.drop_duplicates("iso3").itertuples()},
    "anual": [
        [int(r.anio), int(r.p), r.iso3, redondear(r.fob_usd), redondear(r.peso_neto_kg), redondear(r.fob_p), redondear(r.kg_p)]
        for r in anual_pais.itertuples()
    ],
    "subgrupos": [[int(r.p), r.subgrupo] for r in subgrupos.itertuples()],
    "sub": [
        [int(r.anio), int(r.s), int(r.ue27), redondear(r.fob_usd), redondear(r.peso_neto_kg), redondear(r.fob_p), redondear(r.kg_p)]
        for r in anual_sub.itertuples()
    ],
    "total_ue": [[int(a), redondear(r.fob_total_ue), redondear(r.fob_total_ue_p)] for a, r in total_ue.iterrows()],
}
ruta = WEB_DATA / "datos.json"
ruta.write_text(json.dumps(web, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
print(f"{ruta.relative_to(RAIZ)}: {ruta.stat().st_size / 1e6:.2f} MB · {len(web['anual']):,} filas país-año · {len(web['sub']):,} filas subgrupo")

web\data\datos.json: 0.23 MB · 3,984 filas país-año · 714 filas subgrupo


### Verificación: los totales del JSON cuadran con la tabla de indicadores

In [9]:
chk = anual_pais.assign(ue=anual_pais.iso3.isin(UE27)).query("ue").groupby(["anio", "producto"]).fob_usd.sum()
ref = ind_anual.set_index(["anio", "producto"]).fob_ue
assert (chk.loc[ref.index] - ref).abs().max() < 1, "Los totales del dashboard no cuadran"
print("OK: el FOB a la UE del dashboard coincide con los indicadores anuales")

OK: el FOB a la UE del dashboard coincide con los indicadores anuales
